# 과제 - 신경망을 이용한 손글씨 숫자 인식



## 1. 환경설정



In [ ]:
# Colab: 이 셀을 가장 먼저 실행하세요 (저장소 클론 후 경로·모듈 로드)
# 주의: Colab에서는 GitHub Personal Access Token을 반드시 입력해야 합니다.
import os
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from getpass import getpass

    git_url = "github.com/SISUinSea/Jungle_mnist.git"
    git_branch = "haegeon"
    token = getpass("GitHub Personal Access Token (private 저장소인 경우): ")

    # URL 마지막 경로를 저장소 폴더명으로 사용합니다. (예: .../mnist-lab.git -> mnist-lab)
    repo_name = Path(git_url.rstrip("/")).name
    if repo_name.endswith(".git"):
        repo_name = repo_name[:-4]

    repo_path = Path("/content") / repo_name
    if repo_path.exists():
        os.chdir(repo_path)
        !git fetch origin {git_branch}
        !git checkout {git_branch}
        !git pull --ff-only origin {git_branch}
    else:
        !git clone --branch {git_branch} --single-branch https://{token}@{git_url}
        os.chdir(repo_name)
    sys.path.insert(0, str(Path.cwd() / "src"))
else:
    sys.path.insert(0, "./src")

for module_name in ["activations", "data", "layers", "losses", "network", "optimizers", "training"]:
    sys.modules.pop(module_name, None)


## 2. 데이터 로드

In [ ]:
from data import load_mnist

(x_train, y_train), (x_test, y_test) = load_mnist()
print('Train:', x_train.shape, y_train.shape)
print('Test:', x_test.shape, y_test.shape)

## 3. 구현 및 테스트 통과 확인

`src/` 아래 역할별 파일의 **TODO**를 순서대로 구현한 뒤 아래 셀을 실행하세요.
- 주요 구현 파일: `activations.py`, `layers.py`, `losses.py`, `optimizers.py`, `network.py`, `training.py`
- 구현 파일은 역할별 모듈을 직접 import합니다. 예: `from network import NeuralNetwork`
- 개발 순서: 과제 안내문 참조
- 테스트: `tests/` 아래의 단계별 단위 테스트를 필요한 파일부터 실행합니다. 처음에는 전체 테스트보다 맡은 부분의 테스트 파일을 먼저 실행하세요.
    - ReLU만 확인: `TEST_TARGET = "tests/test_relu.py"`
    - 파일 안의 일부 테스트만 확인: `PYTEST_KEYWORD = "backward"`
    - 전체 테스트 확인: `TEST_TARGET = "tests/"`

In [ ]:
import subprocess
import sys
from pathlib import Path

# Colab/로컬 모두 현재 노트북 실행 위치를 저장소 루트로 사용합니다.
repo_dir = Path.cwd()

# 처음에는 자신이 구현 중인 부분의 테스트 파일만 실행하세요.
# 예: tests/test_relu.py, tests/test_affine.py, tests/test_training.py
TEST_TARGET = "tests/test_relu.py"

# 특정 이름이 들어간 테스트만 실행하고 싶을 때 사용합니다.
# 예: "backward". 전체 파일을 실행하려면 빈 문자열로 둡니다.
PYTEST_KEYWORD = ""

cmd = [sys.executable, "-m", "pytest", TEST_TARGET, "-v"]
if PYTEST_KEYWORD:
    cmd.extend(["-k", PYTEST_KEYWORD])

print("실행 경로:", repo_dir)
print("실행 명령:", " ".join(cmd))
result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    cwd=str(repo_dir)
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode == 0:
    print("\n선택한 테스트를 통과했습니다.")
else:
    print("\n선택한 테스트 중 실패가 있습니다.")


## 4. 모델·옵티마이저 생성 및 학습

In [ ]:
from network import NeuralNetwork
from optimizers import Adam
from training import train

model = NeuralNetwork(use_batchnorm=True, use_dropout=True)  # BatchNorm, Dropout 필수
optimizer = Adam(lr=0.001)

loss_history = train(model, optimizer, x_train, y_train, epochs=20, batch_size=128)

## 5. 평가 및 손실 커브

In [ ]:
from training import evaluate, plot_loss_history

acc, n_params = evaluate(model, x_test, y_test)
print(f'Test Accuracy: {acc:.2f}%')
print(f'Total Params: {n_params:,}')

plot_loss_history(loss_history)

## 6. 은닉층 개수 비교 실험

In [ ]:
import time
import matplotlib.pyplot as plt

from network import NeuralNetwork
from optimizers import Adam
from training import train, evaluate

# 빠른 비교용 설정입니다. 최종 후보는 epochs를 20 이상으로 다시 돌려보세요.
EXPERIMENT_EPOCHS = 5
EXPERIMENT_BATCH_SIZE = 128
EXPERIMENT_LR = 0.001

experiment_configs = [
    {"name": "1-hidden-256", "hidden_sizes": (256,)},
    {"name": "2-hidden-512-256", "hidden_sizes": (512, 256)},
    {"name": "3-hidden-512-256-128", "hidden_sizes": (512, 256, 128)},
]

experiment_results = []
experiment_histories = {}
experiment_models = {}

for config in experiment_configs:
    print(f"\n=== {config['name']} {config['hidden_sizes']} ===")
    model = NeuralNetwork(
        hidden_sizes=config["hidden_sizes"],
        use_batchnorm=True,
        use_dropout=True,
        dropout_ratio=0.5,
    )
    optimizer = Adam(lr=EXPERIMENT_LR)

    start = time.perf_counter()
    loss_history = train(
        model,
        optimizer,
        x_train,
        y_train,
        epochs=EXPERIMENT_EPOCHS,
        batch_size=EXPERIMENT_BATCH_SIZE,
    )
    elapsed = time.perf_counter() - start

    acc, n_params = evaluate(model, x_test, y_test)
    result = {
        "name": config["name"],
        "hidden_sizes": config["hidden_sizes"],
        "accuracy": acc,
        "params": n_params,
        "time_sec": elapsed,
        "final_loss": loss_history[-1],
    }
    experiment_results.append(result)
    experiment_histories[config["name"]] = loss_history
    experiment_models[config["name"]] = model

    print(f"accuracy={acc:.2f}% params={n_params:,} time={elapsed:.2f}s final_loss={loss_history[-1]:.4f}")

print("\nname                 hidden_sizes        acc(%)   params      time(s)  final_loss")
for result in experiment_results:
    print(
        f"{result['name']:<20} {str(result['hidden_sizes']):<18} "
        f"{result['accuracy']:>6.2f} {result['params']:>9,} "
        f"{result['time_sec']:>8.2f} {result['final_loss']:>10.4f}"
    )

plt.figure(figsize=(8, 5))
for name, history in experiment_histories.items():
    plt.plot(history, label=name)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Hidden Layer Count Comparison")
plt.legend()
plt.show()

## 7. 97% 정확도 유지 경량화 실험

은닉층 크기를 줄여가며 97% 이상 정확도를 유지하는 최소 파라미터 수를 탐색합니다. 빠른 탐색은 `LIGHTWEIGHT_EPOCHS = 10`으로 시작하고, 최종 후보는 20 epoch 이상으로 다시 확인하세요.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

from data import load_mnist
from network import NeuralNetwork
from optimizers import Adam
from training import train, evaluate

if "x_train" not in globals() or "x_test" not in globals():
    (x_train, y_train), (x_test, y_test) = load_mnist()

# 97% 경계가 있을 가능성이 높은 구간을 촘촘하게 넣었습니다.
lightweight_configs = [
    {"name": "1-hidden-8", "hidden_sizes": (8,)},
    {"name": "1-hidden-16", "hidden_sizes": (16,)},
    {"name": "1-hidden-32", "hidden_sizes": (32,)},
    {"name": "1-hidden-40", "hidden_sizes": (40,)},
    {"name": "1-hidden-48", "hidden_sizes": (48,)},
    {"name": "1-hidden-56", "hidden_sizes": (56,)},
    {"name": "1-hidden-64", "hidden_sizes": (64,)},
    {"name": "1-hidden-80", "hidden_sizes": (80,)},
    {"name": "1-hidden-128", "hidden_sizes": (128,)},
    {"name": "1-hidden-256", "hidden_sizes": (256,)},
]

LIGHTWEIGHT_EPOCHS = 10
LIGHTWEIGHT_BATCH_SIZE = 128
LIGHTWEIGHT_LR = 0.001
LIGHTWEIGHT_DROPOUT_RATIO = 0.2
LIGHTWEIGHT_SEEDS = [0, 1, 2]
TARGET_ACCURACY = 97.0

lightweight_runs = []

for config in lightweight_configs:
    for seed in LIGHTWEIGHT_SEEDS:
        np.random.seed(seed)

        model = NeuralNetwork(
            hidden_sizes=config["hidden_sizes"],
            use_batchnorm=True,
            use_dropout=True,
            dropout_ratio=LIGHTWEIGHT_DROPOUT_RATIO,
        )
        optimizer = Adam(lr=LIGHTWEIGHT_LR)

        start = time.perf_counter()
        loss_history = train(
            model,
            optimizer,
            x_train,
            y_train,
            epochs=LIGHTWEIGHT_EPOCHS,
            batch_size=LIGHTWEIGHT_BATCH_SIZE,
        )
        elapsed = time.perf_counter() - start

        acc, n_params = evaluate(model, x_test, y_test)
        lightweight_runs.append({
            "name": config["name"],
            "hidden_sizes": config["hidden_sizes"],
            "seed": seed,
            "accuracy": acc,
            "params": n_params,
            "time_sec": elapsed,
            "final_loss": loss_history[-1],
        })

        print(
            f"{config['name']:<13} seed={seed} "
            f"acc={acc:5.2f}% params={n_params:8,} "
            f"time={elapsed:6.1f}s final_loss={loss_history[-1]:.4f}"
        )

summary_rows = []
for config in lightweight_configs:
    rows = [run for run in lightweight_runs if run["name"] == config["name"]]
    accuracies = np.array([row["accuracy"] for row in rows])
    times = np.array([row["time_sec"] for row in rows])
    summary_rows.append({
        "name": config["name"],
        "hidden_sizes": config["hidden_sizes"],
        "params": rows[0]["params"],
        "mean_acc": float(accuracies.mean()),
        "min_acc": float(accuracies.min()),
        "max_acc": float(accuracies.max()),
        "mean_time_sec": float(times.mean()),
        "pass_97_all_seeds": bool(accuracies.min() >= TARGET_ACCURACY),
    })

print("\nname          hidden_sizes   params    mean_acc  min_acc  max_acc  pass_97_all_seeds")
for row in summary_rows:
    print(
        f"{row['name']:<13} {str(row['hidden_sizes']):<13} "
        f"{row['params']:>8,} {row['mean_acc']:>9.2f} "
        f"{row['min_acc']:>8.2f} {row['max_acc']:>8.2f} "
        f"{str(row['pass_97_all_seeds']):>17}"
    )

stable_candidates = [row for row in summary_rows if row["pass_97_all_seeds"]]
if stable_candidates:
    best = min(stable_candidates, key=lambda row: row["params"])
    print(
        f"\n97%를 모든 seed에서 넘긴 최소 모델: "
        f"{best['name']} hidden_sizes={best['hidden_sizes']} "
        f"params={best['params']:,} min_acc={best['min_acc']:.2f}%"
    )
else:
    best = max(summary_rows, key=lambda row: row["mean_acc"])
    print(
        f"\n모든 seed에서 97%를 넘긴 모델은 없습니다. "
        f"가장 높은 평균 정확도 모델: {best['name']} "
        f"mean_acc={best['mean_acc']:.2f}% params={best['params']:,}"
    )

plt.figure(figsize=(8, 5))
plt.plot(
    [row["params"] for row in summary_rows],
    [row["mean_acc"] for row in summary_rows],
    marker="o",
    label="mean accuracy",
)
plt.fill_between(
    [row["params"] for row in summary_rows],
    [row["min_acc"] for row in summary_rows],
    [row["max_acc"] for row in summary_rows],
    alpha=0.2,
    label="seed min-max",
)
plt.axhline(TARGET_ACCURACY, color="red", linestyle="--", label="97% target")
plt.xlabel("Total Parameters")
plt.ylabel("Test Accuracy (%)")
plt.title("Accuracy vs. Parameter Count")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()